# Your research idea → your agent → V6 evidence

This notebook rehearses the workflow using **one fictional opinion and no paid calls**. It creates a fresh folder under the notebook's current directory, uses the installed `termfinder` package, and keeps all outputs in that folder. Run cells in order. No shell cells, credentials, network requests, or external agent session are used.

In a real study you would give V6's request to your own ChatGPT or Claude agent. Here we construct a **scripted surrogate bundle** so the workflow is reproducible offline. This is not a test of an external agent's ability, a research dataset, or an extraction benchmark.

[CARE](https://arxiv.org/abs/2504.05154) motivates a separate research example about culturally specific preferences. Our quiet-room toy is synthetic software verification and does not reproduce CARE or describe a real participant. See the [agent workflow guide](../../docs/AGENT_WORKFLOW.md) for the actual research starting points.


In [ ]:
from copy import deepcopy
import hashlib
import json
from pathlib import Path
import tempfile

from termfinder import FakeProvider, Project, Response
from termfinder.agent_setup import write_handoff

# A new directory on every run; existing studies and exports are preserved.
workspace = Path(tempfile.mkdtemp(prefix="v6-agent-notebook-", dir=Path.cwd())).resolve()
corpus = workspace / "corpus"
corpus.mkdir()
source_path = corpus / "fictional-note.md"
opinion = "I prefer the quiet room because it helps me focus."
toy_text = (
    "# Fictional planning note\n\n"
    "Mira wrote, “" + opinion + "”\n\n"
    "This note is synthetic and does not describe a real participant.\n"
)
source_path.write_text(toy_text, encoding="utf-8")
project = Project(workspace / "study")
print("Fresh local rehearsal folder:", workspace)


## 1. Tell V6 what you want

Choose **Opinion** and explain the scope in ordinary language. The real browser form supplies these fields without asking you to write JSON. We use the API here so a notebook or a student's own coding agent can do the same preparation.

The source root is our newly created toy folder, the maximum is one source, and both the profile and spending limit are offline. The brief is a human-controlled boundary; the returned plan cannot widen it.


In [ ]:
brief = {
    "schema_version": 1,
    "id": "notebook-opinion-rehearsal",
    "data_type": "opinion",
    "starter_recipe": "jargon",
    "research_question": "Which preferences about working spaces are explicitly expressed in this fictional note?",
    "inclusion_criteria": [
        "Collect an exact contiguous passage explicitly expressing a preference about a working space.",
        "Keep its attribution and surrounding source context; interpretations remain suggestions.",
    ],
    "exclusion_criteria": [
        "Exclude factual descriptions without an expressed preference.",
        "Do not infer a real person's identity, endorsement or population opinion from this synthetic source.",
    ],
    "languages": ["en"],
    "communities": ["synthetic-working-space"],
    "domains": ["local-only.invalid"],
    "local_roots": [str(corpus)],
    "budget_usd": 0,
    "mode": "broad",
    "model_profile": "fake",
    "max_sources": 1,
    "dates": {"start": None, "end": None, "unknown": "include"},
}
request = project.prepare_agent_setup(brief)
handoff = write_handoff(request, workspace / "handoff")
assert project.inventory()["sources"] == []  # Preparation reads no source content.
print("Handoff files:", handoff["files"])
print("Copy handoff/prompt.md to your own agent in a real study.")
print("Request preview:\n", request["prompt"][:800], "\n…")


## 2. Receive the plan from your own agent

A real agent returns one bundle containing the recipe, source list, source metadata, sampling notes, and evaluation plan. V6 will turn that bundle into the separate collector input files for you.

The next cell is a **simulated returned bundle**, copied from V6's template and filled with our declared toy source. It is deliberately visible instead of pretending that ChatGPT or Claude was tested. The synthetic English examples explain the exact-span contract; real agents must adapt examples and the annotation rubric to the actual domain and languages.


In [ ]:
bundle = deepcopy(request["starter_bundle"])
bundle["unresolved_questions"] = []  # The tiny toy policy is fully stated above.
bundle["agent"] = {
    "tool": "scripted_notebook_surrogate",
    "model": None,
    "session_reference": None,
    "generated_at": None,
}
bundle["sampling"] = {
    "unit": "source",
    "strategy": "Use the one synthetic note created by this notebook.",
    "stopping_rule": "Stop after the declared one-source toy corpus.",
    "limitations": [
        "Synthetic convenience fixture; not a real population or external-agent source search.",
        "One family may yield an empty held-out split; no research recall claim is made.",
    ],
}
bundle["search_log"] = []  # No search happened; do not fabricate one.
bundle["sources"] = [{
    "source_key": "fictional-note",
    "kind": "local",
    "locator": str(source_path),
    "metadata": {
        "language": "en",
        "community": "synthetic-working-space",
        "domain": "synthetic-opinion",
        "publication_date": None,
        "publication_date_reason": "synthetic_fixture_without_publication_date",
        "constraints": {
            "content_origin": "synthetic_notebook_fixture",
            "license": "Synthetic text created for this local demonstration.",
            "permitted_use": "Offline software verification only; no research observation claimed.",
            "redistribution": "Toy text only; not a third-party source permission assertion.",
            "original_bytes_archive": str(source_path),
        },
    },
    "why_selected": "Declared fictional source for demonstrating an observed opinion span.",
    "origin": "researcher_supplied",
    "search_refs": [],
    "family_id": "fictional-note-family",
}]
recipe = bundle["recipe"]
recipe["execution"].update(concurrency=1, max_calls=4, deadline_seconds=120)
recipe["examples"] = {
    "positive": [{
        "text": "I like the sunny desk.", "expression": "I like the sunny desk.",
        "language": "en", "sense": "A preference for a sunny desk.",
        "reason": "Synthetic positive example: explicit preference; keep observed wording.",
    }],
    "negative": [{
        "text": "The office contains four desks.", "expression": "four desks",
        "language": "en", "sense": None,
        "reason": "Synthetic negative example: a factual description without an expressed preference.",
    }],
    "boundary": [{
        "text": "A fictional novel says, 'Some people prefer a quiet room.'",
        "expression": "Some people prefer a quiet room.", "language": "en",
        "sense": "A quoted preference; endorsement is not established.",
        "reason": "Synthetic boundary example: retain attribution; a quotation does not establish endorsement.",
    }],
}
(workspace / "returned-bundle.json").write_text(
    json.dumps(bundle, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)


## 3. Check and apply the plan

Validation checks the shape, fixed scope, source path, profile, and finite limits. It does not decide whether a source is representative or an opinion interpretation is correct. For an invalid real bundle, give `repair_prompt` back to your own agent instead of weakening the checks.

Applying the plan imports eligible local sources and creates `study/agent-inputs/<bundle_hash>/`. It does not call models. A remote-source fetch would be a separate explicit step; this plan has none.


In [ ]:
validation = project.validate_agent_bundle(bundle)
assert validation["valid"], validation["errors"]
applied = project.apply_agent_bundle(bundle)
assert applied["applied"]
assert len(project.inventory()["sources"]) == 1
assert project.inventory()["jobs"] == []
print(json.dumps({
    "checked_plan": validation["summary"],
    "collector_input_directory": applied["input_files"]["out"],
    "ingested_sources": len(applied["source_ids"]),
}, ensure_ascii=False, indent=2))


## 4. Prepare independent whole-source review before extraction

`freeze_agent_corpus` saves the corpus, source-family assignments, and blank full-source/held-out packets together. It must run **before any extraction job** for those text hashes. Read the reported split: a one-family toy may have no held-out sources.

These packets are not gold labels. A real study needs independent humans to annotate every eligible span in each complete held-out source under a fixed rubric. V6 cannot verify whether they saw model candidates elsewhere. We do not manufacture human gold or report extraction accuracy in this toy.


In [ ]:
review = project.freeze_agent_corpus("toy-corpus-v1", seed="v6-notebook-toy-v1")
review_directory = Path(review["out"])
blank_packets = [
    json.loads(line)
    for line in (review_directory / "annotation_sources.jsonl").read_text(encoding="utf-8").splitlines()
]
assert len(blank_packets) == 1
assert blank_packets[0]["complete"] is False
assert blank_packets[0]["records"] == []
print(json.dumps({
    "corpus_version_id": review["version_id"],
    "review_directory": review["out"],
    "source_families": review["family_map"],
    "split_counts": {name: len(ids) for name, ids in review["split_manifest"]["splits"].items()},
    "annotation_files": review["annotation_files"],
    "note": review["note"],
}, ensure_ascii=False, indent=2))


## 5. Collect one job, then reopen and resume

The fake provider below returns a scripted exact opinion passage from the toy note. Its interpretation is deliberately a different, generated explanation. The second extractor returns no candidate: V6 must still retain the grounded discovery from the first extractor.

Stopping after one job leaves durable pending work. Reopening the same project demonstrates a bounded interruption/resume workflow without forcing a crash or making a paid request. This scripted result establishes mechanics, not real-model performance.


In [ ]:
frozen_source = project.inventory()["sources"][0]
interpretation = "The fictional speaker expresses a preference for a quiet workspace."
candidate = {
    "expression": opinion,
    "language": "en",
    "sense": interpretation,
    "context": frozen_source["text"],
    "start": None,  # The deterministic checker can locate the exact source occurrence.
    "definition_quote": None,
    "attributes": {},
    "translation": None,
}
provider = FakeProvider(expectations=[], responses=[
    Response(json.dumps({"candidates": [candidate], "coverage_complete": True}, ensure_ascii=False),
             40, 40, "stop", "synthetic-notebook-v1"),
    Response(json.dumps({"candidates": [], "coverage_complete": True}),
             40, 40, "stop", "synthetic-notebook-v1"),
])
assert provider.paid is False
project.collect(provider=provider, max_jobs=1)
checkpoint = project.status()
assert len(provider.calls) == 1
assert checkpoint["jobs"].get("pending", 0) >= 1
print("Checkpoint after one offline job:", checkpoint["jobs"])

# A new Project object reads the durable state from the same SQLite project.
project = Project(workspace / "study")
project.resume(provider=provider)
assert len(provider.calls) == 2
assert project.status()["jobs"].get("pending", 0) == 0
assert len(project.inventory()["records"]) == 1
assert project.inventory()["terms"] == []  # Opinions are not presented as lexical terms.
print("Resumed jobs:", project.status()["jobs"])


## 6. Follow the evidence and undo an annotation

The observed passage is an exact substring of the frozen source. Its interpretation is a model-generated suggestion, even though the provider is synthetic here. Occurrence grounding does not prove semantic correctness, factual truth, or a real person's opinion.

We add and undo a **simulated review event**. The record remains in the complete inventory, and the inactive annotation remains in history. In a real study the researcher would make that judgment after inspecting the evidence.


In [ ]:
inventory = project.inventory()
record = inventory["records"][0]
evidence = inventory["occurrences"][0]
source = next(row for row in inventory["sources"] if row["source_id"] == evidence["source_id"])
assert source["text"][evidence["start"]:evidence["end"]] == record["observed_text"] == opinion
assert hashlib.sha256(source["text"].encode("utf-8")).hexdigest() == source["text_sha256"]
assert record["interpretation"] == interpretation
assert record["interpretation"] != record["observed_text"]
assert record["interpretation_origin"] == "model_generated"
assert len(evidence["provenance"]) == 1  # Single-extractor discoveries survive.
print(json.dumps({
    "record_type": record["record_type"],
    "observed_text": record["observed_text"],
    "interpretation": record["interpretation"],
    "interpretation_origin": record["interpretation_origin"],
    "source_id": evidence["source_id"],
    "text_sha256": source["text_sha256"],
    "unicode_span": [evidence["start"], evidence["end"]],
    "original_context": evidence["context"],
    "content_origin": source["constraints"]["content_origin"],
}, ensure_ascii=False, indent=2))

annotation_id = project.annotate(
    record["record_id"], "approved",
    note="Simulated notebook review event; not an actual human research annotation.",
    actor="synthetic-notebook-reviewer",
)
assert project.inventory()["records"][0]["review_status"] == "approved"
project.undo_annotation(annotation_id)
assert project.inventory()["records"][0]["review_status"] == "unreviewed"
assert len(project.inventory()["records"]) == 1
assert next(a for a in project.inventory()["annotations"] if a["annotation_id"] == annotation_id)["active"] is False
print("Annotation undone; the evidence remains in the full inventory.")


## 7. Save a version and check its publication evidence

Use the returned **version ID**, not its display name, for audits and exports. The audit checks frozen consistency and source links; it does not certify publication acceptance, recall, rights, or correct opinion interpretation.

This notebook preserves its synthetic original file and records its archive location. The audit correctly treats that location as an unverified declaration: it reads frozen JSON, not the original archive. There are no independent human gold labels for this toy.


In [ ]:
version_id = project.snapshot("toy-release-v1")
audit = project.publication_audit(version_id)
assert audit["integrity_passed"], audit["issues"]
assert audit["checks"]["verified_evidence_spans"] == 1
print(json.dumps({
    "version_id": version_id,
    "integrity_passed": audit["integrity_passed"],
    "requires_researcher_review": audit["requires_researcher_review"],
    "issues": audit["issues"],
    "limitations": audit["limitations"],
}, ensure_ascii=False, indent=2))


## 8. Export the exact same saved records twice

JSONL retains native nested values and Unicode. CSV and Parquet have documented encodings in `schema.json`. With the pinned Parquet writer, both export directories should contain identical bytes. Exporting makes no provider call.

The approved subset is empty because we undid the simulated approval. The complete grounded opinion record still appears in `records.jsonl`, and its original context remains in `evidence.jsonl`.


In [ ]:
formats = ["csv", "jsonl", "parquet", "annotation"]
calls_before_export = len(provider.calls)
export_a = project.export(version_id, workspace / "export-a", formats=formats)
export_b = project.export(version_id, workspace / "export-b", formats=formats)

def export_hashes(directory):
    directory = Path(directory)
    return {
        file.name: hashlib.sha256(file.read_bytes()).hexdigest()
        for file in sorted(directory.iterdir()) if file.is_file()
    }

hashes_a = export_hashes(workspace / "export-a")
hashes_b = export_hashes(workspace / "export-b")
assert hashes_a == hashes_b
assert len(provider.calls) == calls_before_export
exported_record = json.loads((workspace / "export-a" / "records.jsonl").read_text(encoding="utf-8"))
assert exported_record["observed_text"] == opinion
assert exported_record["interpretation_origin"] == "model_generated"
assert (workspace / "export-a" / "approved_records.jsonl").read_text(encoding="utf-8") == ""

rehearsal_report = {
    "workflow": "scripted synthetic notebook rehearsal",
    "paid_provider_calls": 0,
    "scripted_fake_calls": len(provider.calls),
    "observed_records": len(project.inventory()["records"]),
    "verified_evidence_spans": audit["checks"]["verified_evidence_spans"],
    "export_files": len(hashes_a),
    "repeated_export_bytes_identical": hashes_a == hashes_b,
    "external_agent_tested": False,
    "research_quality_claim": None,
    "research_quality_claim_reason": "Toy workflow verification with no independent human gold or real model extraction.",
    "workspace": str(workspace),
}
(workspace / "notebook-report.json").write_text(
    json.dumps(rehearsal_report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(rehearsal_report, ensure_ascii=False, indent=2))


## Continue with your actual research

1. Describe your real question and choose Terms, Ideas, Opinions, or Sentences in the local browser.
2. Give the generated request to your own ChatGPT or Claude agent. Bring back its complete bundle and repair validation errors with the provided correction request.
3. Review the actual source list, policy, permissions, finite budget, and sampling limitations. Apply the plan and explicitly fetch any reviewed remote sources.
4. Inspect parsed sources, preserve lawful original files separately, and prepare blind full-source review **before extraction jobs**. Identify source families and inspect the held-out split.
5. Arrange independent complete-source annotation, then authorize the concrete provider run. Inspect meanings and other suggestions separately from observed evidence.
6. Evaluate against the fixed gold/split, save a version, inspect the audit, and export that saved version.

The [workflow guide](../../docs/AGENT_WORKFLOW.md) contains research-informed starting conversations. The [publication protocol](../../docs/PUBLICATION_PROTOCOL.md) explains the evidence package and reviewer questions. No result in this synthetic notebook establishes real-world precision, recall, representativeness, or semantic truth.
